# Regresion Logistica Titanic

Clasificacion con regresion logistica en Titanic.

Conversion conceptual 1:1 desde el ejemplo R homologo.


In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()


def find_data(filename: str) -> Path:
    candidates = [NOTEBOOK_DIR / filename, NOTEBOOK_DIR / "data" / filename]
    for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        candidates.append(parent / filename)
        candidates.append(parent / "data" / filename)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro el archivo de datos: {filename}")

print("Notebook dir:", NOTEBOOK_DIR)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

data_path = find_data("titanic.csv")
df = pd.read_csv(data_path)
y = df["Survived"]
X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["Age", "Fare", "SibSp", "Parch", "Pclass"]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), ["Sex", "Embarked"]),
])

model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))
